In [6]:
from tqdm import tqdm
import glob
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import dask.dataframe as dd

In [ ]:
base_path = '../data/raw/orbis/orbis_pat'

folders = [
    'patents_treated',
    'patents_nontreated'
    ]

files = []
for f in folders:
    files.extend(glob.glob(os.path.join(base_path, f, "*.xlsx")))

print(f"Found {len(files)} files")

In [ ]:
output_dir = '../data/interim'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'orbis_pat_data.parquet')

first_file = True


for file in tqdm(files, desc='processing files'):
    try:
        date_cols = ['Publication date', 'Application/filing date', 'Priority date']

        df = pd.read_excel(file, sheet_name='Results', dtype=str)

        # Forward fill
        ffill_cols = [
            'Publication number',
            'Publication date',
            'Priority date',
            'Number of forward citations',
            'Application number',
            'Application/filing date'
        ]
        ffill_cols = [c for c in ffill_cols if c in df.columns]
        if 'Unnamed: 0' in df.columns and ffill_cols:
            mask = df['Unnamed: 0'].isna()
            if mask.any():
                df_ffilled = df[ffill_cols].ffill()
                df.loc[mask, ffill_cols] = df_ffilled.loc[mask, ffill_cols]

        # rename columns
        df = df.rename(columns={
            "Applicant(s) BvD ID Number(s)": "bvd_id",
            "Publication number": "publ_number",
            "Application number": "appl_number",
            "Publication date": "publ_date",
            "Application/filing date": "appl_date",
            "Priority date": "prio_date",
            'Number of forward citations': 'frw_cit'
        })
        
        # File tracing
        df["source_file"] = os.path.basename(file)

        # Append to Parquet incrementally
        if first_file:
            df.to_parquet(output_file, index=False)
            first_file = False
        else:
            df.to_parquet(output_file, index=False, append=True)

    except Exception as e:
        print(f"error in {os.path.basename(file)}: {e}")

print("all files processed and saved.")

In [4]:
columns_to_keep = [
    'publ_number', 'publ_date', 'prio_date', 'IPC code (main)',
    'frw_cit', 'bvd_id', 'appl_date', 'appl_number'
]

ddf = dd.read_parquet(
    '../data/interim/orbis_pat_data.parquet',
    columns=columns_to_keep
)
ddf = ddf.rename(columns={'IPC code (main)': 'ipc_code'})

# drop duplicates
ddf = ddf.drop_duplicates(keep='first')

ddf.to_parquet(
    '../data/interim/orbis_pat_data_dedupl.parquet',
    engine='pyarrow',
    write_index=False
)

In [5]:
ddf = dd.read_parquet('../data/interim/orbis_pat_data_dedupl.parquet')

pat_orbis = ddf.compute()
print('loaded df shape:', pat_orbis.shape)

loaded df shape: (59012565, 8)


In [10]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def extract_year_chunk(s):
    """Vectorized extraction for a Series."""
    year = pd.Series(np.nan, index=s.index)

    # Numeric values
    numeric = pd.to_numeric(s, errors='coerce')
    mask_year = (numeric >= 1000) & (numeric <= 2100)
    year[mask_year] = numeric[mask_year].astype(int)

    # Excel serial dates
    mask_excel = numeric.notna() & ~mask_year
    if mask_excel.any():
        excel_dates = pd.Timestamp("1899-12-30") + pd.to_timedelta(numeric[mask_excel], unit='D')
        year[mask_excel] = excel_dates.dt.year

    # Date strings
    mask_str = year.isna() & s.notna()
    if mask_str.any():
        parsed_dates = pd.to_datetime(s[mask_str], errors='coerce')
        year[mask_str] = parsed_dates.dt.year

    return year

def process_in_chunks(df, col, chunk_size=500_000):
    """Process a column in chunks to save memory."""
    n = len(df)
    results = []

    for start in tqdm(range(0, n, chunk_size), desc=f"Processing {col}"):
        end = min(start + chunk_size, n)
        chunk = df.iloc[start:end, :].copy()
        chunk_year = extract_year_chunk(chunk[col])
        results.append(chunk_year)

    return pd.concat(results)

In [11]:
for col in ['publ_date', 'appl_date', 'prio_date']:
    new_col = col.replace('_date', '_year')
    pat_orbis[new_col] = process_in_chunks(pat_orbis, col)

Processing prio_date:  79%|███████▉  | 94/119 [00:34<00:09,  2.77it/s]C:\Users\nicol\AppData\Local\Temp\ipykernel_10388\2939042113.py:23: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed_dates = pd.to_datetime(s[mask_str], errors='coerce')
Processing prio_date:  82%|████████▏ | 98/119 [00:35<00:07,  2.77it/s]C:\Users\nicol\AppData\Local\Temp\ipykernel_10388\2939042113.py:23: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed_dates = pd.to_datetime(s[mask_str], errors='coerce')
Processing prio_date: 100%|██████████| 119/119 [00:43<00:00,  2.75it/s]


In [18]:
from collections import defaultdict

df = pat_orbis
cols_to_count = ['publ_number', 'prio_date', 'appl_number', 'ipc_code', 'frw_cit']

counts_dict = defaultdict(lambda: defaultdict(int))

chunk_size = 1_000_000
num_chunks = (len(df) + chunk_size - 1) // chunk_size

for start in tqdm(range(0, len(df), chunk_size), total=num_chunks, desc="processing chunks"):
    chunk = df.iloc[start:start + chunk_size]

    for year_col in ['publ_year', 'appl_year', 'prio_year']:
        grouped = chunk.groupby(['bvd_id', year_col])[cols_to_count].count()
        for (bvd, year), row in grouped.iterrows():
            for col in cols_to_count:
                counts_dict[(bvd, year)][col] += row[col]

# convert dictionary to df
rows = []
for (bvd, year), col_counts in counts_dict.items():
    row = {'bvd_id': bvd, 'year': year}
    row.update({f'{col}_count': col_counts.get(col, 0) for col in cols_to_count})
    rows.append(row)

counts_df = pd.DataFrame(rows).sort_values(['bvd_id', 'year']).reset_index(drop=True)

print(counts_df)

processing chunks: 100%|██████████| 60/60 [26:25<00:00, 26.43s/it]


                  bvd_id    year  publ_number_count  prio_date_count  \
0        AE*110006746731  2008.0                  1                0   
1        AE*110006746731  2010.0                  1                0   
2        AE*110292331894  2014.0                  4                4   
3        AE*110292331894  2015.0                  4                4   
4        AE*110292331894  2016.0                  2                2   
...                  ...     ...                ...              ...   
4030317        ZW30003KZ  2013.0                  1                1   
4030318        ZW30043KZ  2013.0                  1                1   
4030319        ZW30043KZ  2016.0                  2                2   
4030320       ZWFEB16063  2017.0                  2                0   
4030321       ZWFEB16063  2019.0                  2                0   

         appl_number_count  ipc_code_count  frw_cit_count  
0                        1               1              1  
1              

In [20]:
counts_df.to_csv('../data/interim/pat_orbis_counts.csv', index=False)